# 几何平均数（沪深300指数）

> 对应代码：`MathStatsCode/code_in_notes/Chap.7/geometric_mean.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：用沪深300指数数据计算几何平均数 exp(mean(log(增长率)))，并检验首末两天指数之间的关系。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.7`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.7")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

载入沪深300指数数据（含日期与收盘指数）。


In [ ]:
%%stata
use ../datasets/hs300index.dta, clear


**按照时间排序，用行号作为新的时间**

`sort day` 按日期排序后用行号生成时间变量 `t=_n`，再用 `tsset t` 把数据声明为**时间序列**——只有声明之后才能使用滞后算子 `L.` 等时间序列功能。


In [ ]:
%%stata
sort day
gen t=_n
tsset t


**计算增长率**

计算收益率与几何平均数：`gen r=clsindex/L.clsindex` 中 `L.` 是滞后一期算子，r 为相邻两期的**价格比**（增长因子）；取对数 `log_r=log(r)` 后 `su` 求均值，用 `exp(r(mean))` 还原成几何平均增长率——这正是几何平均数的定义：n 期增长因子的连乘开 n 次方等于对数增长率的算术平均再取指数。最后用三条 `di` 验证：首日指数 × 几何平均数的 (N-1) 次方 ≈ 最后一天指数，说明用几何平均数可以准确刻画指数序列的平均增长速度（算术平均会高估）。


In [ ]:
%%stata
gen r=clsindex/L.clsindex
gen log_r=log(r)
su log_r
local geo_mean=exp(r(mean))
di "几何平均数=`geo_mean'"
di "第一天沪深300指数=" clsindex[1]
di "最后一天沪深300指数=" clsindex[_N]
di "使用几何平均数计算：" clsindex[1]*(`geo_mean'^(_N-1))
